In [1]:
import os
import re
import json
import string
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import clone_model
from tensorflow.keras.applications import EfficientNetV2B0
from sklearn.metrics import accuracy_score, classification_report
from transformers import AutoModel, TFAutoModel, AutoTokenizer, BertTokenizer
from PIL import Image
import pandas as pd
from sklearn.model_selection import train_test_split

2025-08-16 14:48:37.291441: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv('click-id/dataset_with_image/combined_final_final.csv')
def get_first_and_last_words(text, first_n=64, last_n=64):
    words = text.split()
    if len(words) <= first_n + last_n:
        return text  
    first_words = words[:first_n]
    last_words = words[-last_n:]
    return ' '.join(first_words + last_words)

df['content'] = df['content'].apply(
    lambda x: get_first_and_last_words(x, 64, 64)
)

In [3]:
tokenizer = BertTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

In [4]:
df_train, df_val = train_test_split(df[['title','content','image_path_processed','label_score']],test_size = 0.2 ,random_state = 93)
df_train.shape , df_val.shape

((11174, 4), (2794, 4))

In [5]:
tokens_train = {
   'title':tokenizer.batch_encode_plus(
    df_train['title'].tolist(),
    max_length = 32,
    padding = 'max_length',
    truncation = True
   ),
    'content':tokenizer.batch_encode_plus(
    df_train['content'].tolist(),
    max_length = 128,
    padding = 'max_length',
    truncation = True
   )
}
tokens_val = {
  'title' : tokenizer.batch_encode_plus(
    df_val['title'].tolist(),
    max_length = 32,
    padding = 'max_length',
    truncation=True
  ),
  'content' : tokenizer.batch_encode_plus(
    df_val['content'].tolist(),
    max_length = 128,
    padding = 'max_length',
    truncation=True
  )
}

In [6]:
IMG_HEIGHT = 224
IMG_WIDTH = 224

def load_and_preprocess_images(image_paths):
    images = []
    n = 0
    for path in image_paths:
        n = n+1
        newpath = "/".join(i for i in path.split("/")[-4:])
        # print(newpath)
        img = tf.io.read_file(newpath)
        img = tf.image.decode_image(img, channels=3)
        # img = tf.cast(img, tf.float32)
        # img = img/255.0

        images.append(img)
        if n%100 == 0:
          print(n,'images processed')
    return tf.stack(images)

with tf.device("/cpu:0"):
    train_images = load_and_preprocess_images(df_train['image_path_processed'].tolist())
    val_images = load_and_preprocess_images(df_val['image_path_processed'].tolist())

2025-08-16 14:49:08.039803: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:49:08.116123: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:49:08.116169: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:49:08.118645: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:984] could not open file to read NUMA node: /sys/bus/pci/devices/0000:05:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-08-16 14:49:08.118687: I external/local_xla/xla/stream_executor

100 images processed
200 images processed
300 images processed
400 images processed
500 images processed
600 images processed
700 images processed
800 images processed
900 images processed
1000 images processed
1100 images processed
1200 images processed
1300 images processed
1400 images processed
1500 images processed
1600 images processed
1700 images processed
1800 images processed
1900 images processed
2000 images processed
2100 images processed
2200 images processed
2300 images processed
2400 images processed
2500 images processed
2600 images processed
2700 images processed
2800 images processed
2900 images processed
3000 images processed
3100 images processed
3200 images processed
3300 images processed
3400 images processed
3500 images processed
3600 images processed
3700 images processed
3800 images processed
3900 images processed
4000 images processed
4100 images processed
4200 images processed
4300 images processed
4400 images processed
4500 images processed
4600 images process

2025-08-16 14:49:18.543548: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1681999872 exceeds 10% of free system memory.


100 images processed
200 images processed
300 images processed
400 images processed
500 images processed
600 images processed
700 images processed
800 images processed
900 images processed
1000 images processed
1100 images processed
1200 images processed
1300 images processed
1400 images processed
1500 images processed
1600 images processed
1700 images processed
1800 images processed
1900 images processed
2000 images processed
2100 images processed
2200 images processed
2300 images processed
2400 images processed
2500 images processed
2600 images processed
2700 images processed


In [7]:
with tf.device("/cpu:0"):
    def save_tf_dataset(images, filename):
        dataset = tf.data.Dataset.from_tensor_slices(images)
        
        tf.data.Dataset.save(dataset, filename)
        print(f"Dataset saved to {filename}")
    
    # Save the datasets
    save_tf_dataset(train_images, "./train_half_images_dataset")
    save_tf_dataset(val_images, "./val_half_images_dataset")

2025-08-16 14:49:20.843165: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1681999872 exceeds 10% of free system memory.


Dataset saved to ./train_half_images_dataset
Dataset saved to ./val_half_images_dataset


In [8]:
train_images = tf.data.Dataset.load("train_half_images_dataset")
val_images = tf.data.Dataset.load("val_half_images_dataset")

In [9]:
train_data = {
    'title':tf.squeeze(tf.convert_to_tensor(tokens_train['title']['input_ids'])),
    'content':tf.squeeze(tf.convert_to_tensor(tokens_train['content']['input_ids'])),
    'titlemask':tf.squeeze(tf.convert_to_tensor(tokens_train['title']['attention_mask'])),
    'contentmask':tf.squeeze(tf.convert_to_tensor(tokens_train['content']['attention_mask'])),
}

validation_data = {
    'title':tf.convert_to_tensor(tokens_val['title']['input_ids']),
    'content':tf.convert_to_tensor(tokens_val['content']['input_ids']),
    'titlemask':tf.convert_to_tensor(tokens_val['title']['attention_mask']),
    'contentmask':tf.convert_to_tensor(tokens_val['content']['attention_mask']),
}


label_train = tf.convert_to_tensor(df_train['label_score'].tolist())
label_val = tf.convert_to_tensor(df_val['label_score'].tolist())

In [10]:
dataset = (tf.data.Dataset.from_tensor_slices((dict(train_data),label_train))
    .prefetch(tf.data.AUTOTUNE)
    .cache()
    )
dataVal = (tf.data.Dataset.from_tensor_slices((dict(validation_data),label_val))
    .prefetch(tf.data.AUTOTUNE)
    .cache()
    )

dataset = tf.data.Dataset.zip((dataset, train_images))
dataVal = tf.data.Dataset.zip((dataVal, val_images))

def add_image_and_fix_shapes(text_data_label, image):
    text_data, label = text_data_label
    new_data = {
        'title': tf.ensure_shape(text_data['title'], [None, 32]),
        'content': tf.ensure_shape(text_data['content'], [None, 128]),
        'titlemask': tf.ensure_shape(text_data['titlemask'], [None, 32]),
        'contentmask': tf.ensure_shape(text_data['contentmask'], [None, 128]),
        'image': image
    }
    
    return new_data, label

dataset = (dataset
    .batch(32)
    .map(add_image_and_fix_shapes)
    .prefetch(tf.data.AUTOTUNE)
    .cache()
)

dataVal = (dataVal
    .batch(32)
    .map(add_image_and_fix_shapes)
    .prefetch(tf.data.AUTOTUNE)
    .cache()
)

In [11]:
tf.data.Dataset.save(dataset, "./half_dataset")
tf.data.Dataset.save(dataVal, "./half_dataVal")

In [12]:
dataset = tf.data.Dataset.load(
    "half_dataset",
    element_spec=({
        'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
    }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
)

dataVal = tf.data.Dataset.load(
    "half_dataVal",
    element_spec=({
        'title': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'content': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'titlemask': tf.TensorSpec(shape=(None, 32), dtype=tf.int32),
        'contentmask': tf.TensorSpec(shape=(None, 128), dtype=tf.int32),
        'image': tf.TensorSpec(shape=(None, 224, 224, 3), dtype=tf.uint8)
    }, tf.TensorSpec(shape=(None,), dtype=tf.int32))
)

In [13]:
class BertEmbeddingLayer(tf.keras.layers.Layer):
    def __init__(self, model_name, **kwargs):
        super(BertEmbeddingLayer, self).__init__(**kwargs)
        self.model_name = model_name
        self.embedding_size = None
        self.bert = None

    def build(self, input_shape):
        self.bert = TFAutoModel.from_pretrained(self.model_name)
        self.bert.trainable = True
        self.embedding_size = self.bert.config.hidden_size
        super(BertEmbeddingLayer, self).build(input_shape)

    def call(self, inputs):
        ids, att = inputs
        outputs = self.bert(input_ids=ids, attention_mask=att)
        return outputs.pooler_output

    def get_config(self):
        config = super(BertEmbeddingLayer, self).get_config()
        config.update({
            "model_name": self.model_name
        })
        return config
    
    @classmethod
    def from_config(cls, config):
        clean_config = {k: v for k, v in config.items() 
                      if k in ['model_name', 'name', 'trainable', 'dtype']}
        return cls(**clean_config)

In [14]:
def makemodel(output_bias=None):
    # Text using IndoBertp1
    ids1 = tf.keras.layers.Input(shape=(32,), dtype=tf.int32, name="title")
    att1 = tf.keras.layers.Input(shape=(32,), dtype=tf.int32, name="titlemask")
    ids2 = tf.keras.layers.Input(shape=(128,), dtype=tf.int32, name="content")
    att2 = tf.keras.layers.Input(shape=(128,), dtype=tf.int32, name="contentmask")

    indobert1 = BertEmbeddingLayer(model_name="indobenchmark/indobert-base-p1", name="indobert1")
    indobert2 = BertEmbeddingLayer(model_name="indobenchmark/indobert-base-p1", name="indobert2")

    title = indobert1([ids1, att1])
    content = indobert2([ids2, att2])

    # Image using EfficientNetV2s
    img1 = tf.keras.layers.Input(shape=(224,224,3), dtype=tf.uint8, name="image")

    effi = EfficientNetV2B0(
        include_top = False,
        weights = 'imagenet',
        input_shape = (224,224,3),
        pooling = 'max'
    )
    image = effi(img1)

    fin = tf.keras.layers.concatenate([title, content, image])
    fin = tf.keras.layers.BatchNormalization()(fin)
        
    fin = tf.keras.layers.Dense(512, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 

    fin = tf.keras.layers.Dense(256, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 
    
    fin = tf.keras.layers.Dense(128, activation=None)(fin)
    fin = tf.keras.layers.BatchNormalization()(fin)
    fin = tf.keras.layers.Activation('relu')(fin)
    fin = tf.keras.layers.Dropout(0.4)(fin) 
    
    fin = tf.keras.layers.Dense(64, activation='relu')(fin)
    fin = tf.keras.layers.Dropout(0.2)(fin)
    fin = tf.keras.layers.Dense(1, activation='sigmoid')(fin)

    final = tf.keras.Model(inputs=[ids1, att1, ids2, att2, img1], outputs=fin)

    for layer in final.layers:
      layer.trainable = True
    optimizer = tf.keras.optimizers.Adam(learning_rate = 0.00005)
    final.compile(
        loss = tf.keras.losses.BinaryCrossentropy(),
        optimizer=optimizer,
        metrics=[tf.keras.metrics.BinaryAccuracy(),tf.keras.metrics.Precision(),tf.keras.metrics.Recall()]
    )

    return final

final = makemodel()
final.summary()

Some layers from the model checkpoint at indobenchmark/indobert-base-p1 were not used when initializing TFBertModel: ['mlm___cls', 'nsp___cls', 'bert/encoder/layer_._7/attention/output/dense/kernel:0', 'bert/encoder/layer_._8/attention/self/key/bias:0', 'bert/encoder/layer_._1/attention/self/key/bias:0', 'bert/encoder/layer_._0/attention/self/value/kernel:0', 'bert/encoder/layer_._2/attention/self/key/kernel:0', 'bert/encoder/layer_._6/output/dense/kernel:0', 'bert/encoder/layer_._10/attention/self/value/bias:0', 'bert/encoder/layer_._1/attention/self/query/kernel:0', 'bert/encoder/layer_._8/attention/output/dense/bias:0', 'bert/encoder/layer_._11/output/dense/kernel:0', 'bert/encoder/layer_._7/attention/output/LayerNorm/beta:0', 'bert/encoder/layer_._9/output/LayerNorm/gamma:0', 'bert/encoder/layer_._5/output/dense/kernel:0', 'bert/encoder/layer_._7/attention/self/value/kernel:0', 'bert/encoder/layer_._3/attention/output/dense/bias:0', 'bert/encoder/layer_._7/intermediate/dense/kernel

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ title (InputLayer)  │ (None, 32)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ titlemask           │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content             │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ contentmask         │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ indobert1           │ (None, 768)       │          0 │ title[0][0],      │
│ (BertEmbeddingLaye… │                   │            │ titlemask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ indobert2           │ (None, 768)       │          0 │ content[0][0],    │
│ (BertEmbeddingLaye… │                   │            │ contentmask[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ efficientnetv2-b0   │ (None, 1280)      │  5,919,312 │ image[0][0]       │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 2816)      │          0 │ indobert1[0][0],  │
│ (Concatenate)       │                   │            │ indobert2[0][0],  │
│                     │                   │            │ efficientnetv2-b… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 2816)      │     11,264 │ concatenate[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │  1,442,304 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 512)       │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 256)       │    131,328 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256)       │      1,024 │ dense_1[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 256)       │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 256)       │          0 │ activation_1[0][

 Total params: 7,549,009 (28.80 MB)

 Trainable params: 7,480,977 (28.54 MB)

 Non-trainable params: 68,032 (265.75 KB)

In [15]:
tf.keras.mixed_precision.set_global_policy('mixed_float16')

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
    )

# model_checkpoint = tf.keras.callbacks.ModelCheckpoint(
#     filepath='./models/best_model_val_loss_{val_loss:.4f}.weights.h5',
#     monitor='val_loss',
#     save_best_only=True,
#     save_weights_only=True
# )

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=1,
    min_lr=1e-8
)

history=final.fit(x = dataset,
                  epochs = 50,
                  callbacks=[early_stopping, 
                             # model_checkpoint, 
                             reduce_lr],
                  validation_data=dataVal
                  )

Epoch 1/50


I0000 00:00:1755330646.215112     620 service.cc:145] XLA service 0x72d3540053a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1755330646.215164     620 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3070, Compute Capability 8.6
2025-08-16 14:50:47.850600: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
W0000 00:00:1755330648.739309     620 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755330648.832507     620 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
2025-08-16 14:50:53.863305: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:465] Loaded cuDNN version 8907
I0000 00:00:1755330670.805171    1145 asm_compiler.cc:369] ptxas warning :

349/350 ━━━━━━━━━━━━━━━━━━━━ 0s 131ms/step - binary_accuracy: 0.4778 - loss: 0.9230 - precision: 0.4437 - recall: 0.7862

W0000 00:00:1755330761.975152     619 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755330761.980407     619 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
I0000 00:00:1755330782.329849    1232 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_31', 12 bytes spill stores, 12 bytes spill loads

I0000 00:00:1755330782.525281    1239 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_31', 20 bytes spill stores, 20 bytes spill loads

I0000 00:00:1755330782.716436    1242 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_7', 96 bytes spill stores, 96 bytes spill loads

I0000 00:00:1755330782.820316    1246 asm_compiler.cc:369] ptxas warning : Registers are spilled to l

350/350 ━━━━━━━━━━━━━━━━━━━━ 0s 301ms/step - binary_accuracy: 0.4780 - loss: 0.9226 - precision: 0.4437 - recall: 0.7858

W0000 00:00:1755330824.742629     618 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755330824.838795     618 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
I0000 00:00:1755330827.203558    1492 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_13839', 8 bytes spill stores, 8 bytes spill loads

W0000 00:00:1755330838.069802     622 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755330838.074463     622 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert
I0000 00:00:1755330839.466486    1571 asm_compiler.cc:369] ptxas warning : Registers are spilled to local memory in function 'triton_gemm_dot_2', 12 bytes spil

350/350 ━━━━━━━━━━━━━━━━━━━━ 250s 388ms/step - binary_accuracy: 0.4781 - loss: 0.9223 - precision: 0.4437 - recall: 0.7854 - val_binary_accuracy: 0.6392 - val_loss: 0.6508 - val_precision: 0.6241 - val_recall: 0.3827 - learning_rate: 5.0000e-05
Epoch 2/50
350/350 ━━━━━━━━━━━━━━━━━━━━ 53s 152ms/step - binary_accuracy: 0.5987 - loss: 0.6907 - precision: 0.5403 - recall: 0.4954 - val_binary_accuracy: 0.6858 - val_loss: 0.6043 - val_precision: 0.6785 - val_recall: 0.4971 - learning_rate: 5.0000e-05
Epoch 3/50
350/350 ━━━━━━━━━━━━━━━━━━━━ 56s 159ms/step - binary_accuracy: 0.6358 - loss: 0.6574 - precision: 0.5900 - recall: 0.5234 - val_binary_accuracy: 0.7026 - val_loss: 0.5813 - val_precision: 0.6980 - val_recall: 0.5307 - learning_rate: 5.0000e-05
Epoch 4/50
350/350 ━━━━━━━━━━━━━━━━━━━━ 53s 152ms/step - binary_accuracy: 0.6648 - loss: 0.6198 - precision: 0.6272 - recall: 0.5584 - val_binary_accuracy: 0.7037 - val_loss: 0.5697 - val_precision: 0.6947 - val_recall: 0.5416 - learning_rate: 5

In [16]:
y_pred = final.predict(dataVal)
y_pred_binary = (y_pred > 0.5).astype(int)
from sklearn.metrics import classification_report
y_val = np.concatenate([y for _, y in dataVal.as_numpy_iterator()])
print(classification_report(y_val, y_pred_binary, 
                           target_names=['Class_0', 'Class_1'],
                           digits = 4))

W0000 00:00:1755331520.130578     619 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755331520.160541     619 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert


87/88 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step

W0000 00:00:1755331540.092215     616 assert_op.cc:38] Ignoring Assert operator functional_1/indobert1_1/tf_bert_model/bert/embeddings/assert_less/Assert/Assert
W0000 00:00:1755331540.186112     616 assert_op.cc:38] Ignoring Assert operator functional_1/indobert2_1/tf_bert_model_1/bert/embeddings/assert_less/Assert/Assert


88/88 ━━━━━━━━━━━━━━━━━━━━ 29s 236ms/step
              precision    recall  f1-score   support

     Class_0     0.7651    0.8137    0.7886      1605
     Class_1     0.7249    0.6627    0.6924      1189

    accuracy                         0.7495      2794
   macro avg     0.7450    0.7382    0.7405      2794
weighted avg     0.7480    0.7495    0.7477      2794



2025-08-16 15:05:45.383172: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
